# Experimento principal

## Imports necesarios

In [31]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, hinge_loss
from joblib import Parallel, delayed

## Funciones 

In [32]:
def procesar_dataset(archivo):
    """
    Función objetivo para joblib. Procesa un único dataset completo usando SVC.
    Retorna el nombre del archivo, las métricas, las estadísticas y posibles errores.
    """
    if not os.path.exists(archivo):
        return archivo, None, None, f"[ERROR] Archivo no encontrado: {archivo}"

    df = pd.read_csv(archivo)
    X = df.drop(columns=['clase']).values
    X = df.drop(columns=['clase', 'P_0', 'P_1', 'P_2', 'mu_2', 'sigma_2']).values
    y = df['clase'].values

    # Estructura ajustada sin validación
    metricas = {
        'acc_train': [], 
        'acc_test': [], 
        'recall_test': [], 
        'f1_test': [], 
        'auc_test': []
    }
    
    # Única división: 85% Entrenamiento, 15% Prueba
    sss_test = StratifiedShuffleSplit(n_splits=10, test_size=0.15, random_state=42)

    fold = 1
    for train_idx, test_idx in sss_test.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        svm = SVC(kernel='linear', random_state=42)
        svm.fit(X_train, y_train)

        y_pred_train = svm.predict(X_train)
        y_pred_test = svm.predict(X_test)
        
        y_scores_test = svm.decision_function(X_test)

        metricas['acc_train'].append(accuracy_score(y_train, y_pred_train))
        metricas['acc_test'].append(accuracy_score(y_test, y_pred_test))
        metricas['recall_test'].append(recall_score(y_test, y_pred_test))
        metricas['f1_test'].append(f1_score(y_test, y_pred_test))
        metricas['auc_test'].append(roc_auc_score(y_test, y_scores_test))

        fold += 1

    df_metricas = pd.DataFrame(metricas)
    df_metricas.index = [f"Fold {i+1}" for i in range(10)]

    stats = pd.DataFrame({
        'Media': df_metricas.mean(),
        'Mediana': df_metricas.median(),
        'Desv. Est.': df_metricas.std()
    }).T

    return archivo, df_metricas, stats, None

In [33]:
def entrenar_evaluar_svm_paralelo():
    archivos = [
        "radiomics_DE_rand_1.csv",
        "radiomics_DE_best_1.csv",
        "radiomics_DE_rand_2.csv",
        "radiomics_DE_best_2.csv",
        "radiomics_Standard_Otsu.csv"
    ]

    print(f"[INFO] Iniciando procesamiento en paralelo para {len(archivos)} datasets con joblib...")
    print(f"[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).")

    # Ejecución concurrente
    resultados = Parallel(n_jobs=-1)(delayed(procesar_dataset)(archivo) for archivo in archivos)

    # Impresión de resultados
    for archivo, df_metricas, stats, error in resultados:
        if error:
            print(error)
            continue

        print(f"\n{'-'*60}")
        print(f"[INFO] Resultados Dataset: {archivo}")
        print(f"{'-'*60}")
        print(f"\n[RESULTADOS] 10 Folds:")
        print(df_metricas.round(4).to_string())

        print(f"\n[ESTADISTICAS] Finales ({archivo}):")
        print(stats.round(4).to_string())
        print("\n")

In [34]:
if __name__ == "__main__":
    entrenar_evaluar_svm_paralelo()

[INFO] Iniciando procesamiento en paralelo para 5 datasets con joblib...
[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).

------------------------------------------------------------
[INFO] Resultados Dataset: radiomics_DE_rand_1.csv
------------------------------------------------------------

[RESULTADOS] 10 Folds:
         acc_train  acc_test  recall_test  f1_test  auc_test
Fold 1      0.9307    0.9306       0.9548   0.9525    0.9561
Fold 2      0.9309    0.9352       0.9563   0.9556    0.9582
Fold 3      0.9307    0.9329       0.9548   0.9540    0.9599
Fold 4      0.9333    0.9226       0.9563   0.9474    0.9376
Fold 5      0.9329    0.9147       0.9485   0.9419    0.9418
Fold 6      0.9307    0.9283       0.9501   0.9508    0.9633
Fold 7      0.9303    0.9340       0.9532   0.9547    0.9696
Fold 8      0.9361    0.9044       0.9407   0.9349    0.9374
Fold 9      0.9311    0.9306       0.9516   0.9524    0.9703
Fold 10     0.9321    0.9249       0.9501   0.9486    0.96

In [35]:
def entrenar_evaluar_svm_paralelo():
    archivos = [
    "radiomics_DE_rand_1.csv",
    "radiomics_DE_best_1.csv",
    "radiomics_DE_rand_2.csv",
    "radiomics_DE_best_2.csv",
    "radiomics_Standard_Otsu.csv"
]

    print(f"[INFO] Iniciando procesamiento en paralelo para {len(archivos)} datasets con joblib...")
    print(f"[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).")

    # Ejecución concurrente
    resultados = Parallel(n_jobs=-1)(delayed(procesar_dataset)(archivo) for archivo in archivos)

    resumen_medias = []

    # Impresión individual y recolección de datos
    for archivo, df_metricas, stats, error in resultados:
        if error:
            print(error)
            continue

        print(f"\n{'-'*60}")
        print(f"[INFO] Resultados Dataset: {archivo}")
        print(f"{'-'*60}")
        print(df_metricas.round(4).to_string())
        print(f"\n[ESTADISTICAS]:\n{stats.round(4).to_string()}\n")

        # Extraer la fila de "Media" para la tabla comparativa final
        media_esquema = stats.loc['Media'].copy()
        
        # Limpiar el nombre del archivo para usarlo como etiqueta
        nombre_limpio = archivo.replace('.csv', '').replace('radiomics_', '')
        media_esquema['Esquema'] = nombre_limpio
        
        resumen_medias.append(media_esquema)

    # Construcción y ordenamiento de la tabla global
    if resumen_medias:
        df_final = pd.DataFrame(resumen_medias)
        df_final.set_index('Esquema', inplace=True)
        
        # Ordenar el DataFrame tomando 'acc_test' como métrica principal, descendente
        df_final = df_final.sort_values(by='acc_test', ascending=False)

        print("\n" + "="*80)
        print("[RESULTADO GLOBAL] Comparación de Esquemas (Ordenado por acc_test)")
        print("="*80)
        print(df_final.round(4).to_string())
        
        # Opcional: Exportar la tabla final a CSV
        df_final.to_csv("comparacion_esquemas_DE.csv")
        print("\n[INFO] La tabla global ha sido guardada como 'comparacion_esquemas_DE.csv'")

if __name__ == "__main__":
    entrenar_evaluar_svm_paralelo()

[INFO] Iniciando procesamiento en paralelo para 5 datasets con joblib...
[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).

------------------------------------------------------------
[INFO] Resultados Dataset: radiomics_DE_rand_1.csv
------------------------------------------------------------
         acc_train  acc_test  recall_test  f1_test  auc_test
Fold 1      0.9307    0.9306       0.9548   0.9525    0.9561
Fold 2      0.9309    0.9352       0.9563   0.9556    0.9582
Fold 3      0.9307    0.9329       0.9548   0.9540    0.9599
Fold 4      0.9333    0.9226       0.9563   0.9474    0.9376
Fold 5      0.9329    0.9147       0.9485   0.9419    0.9418
Fold 6      0.9307    0.9283       0.9501   0.9508    0.9633
Fold 7      0.9303    0.9340       0.9532   0.9547    0.9696
Fold 8      0.9361    0.9044       0.9407   0.9349    0.9374
Fold 9      0.9311    0.9306       0.9516   0.9524    0.9703
Fold 10     0.9321    0.9249       0.9501   0.9486    0.9626

[ESTADISTICAS]:
    